# Kuiper kernel benchmarks

Each table compares the JIT-dispatched verified Kuiper kernels (`kuipy.run`) and
the unverified reference kernels (`kuipy.unverified`) against stock PyTorch, on
the shapes of a Qwen2.5-0.5B decode step.

Every GEMM here has a transposed right operand, because that is what the model
emits: `nn.Linear` stores its weight as `(N, K)` and hands ATen a view of it, so
`mm`/`addmm` see a `(K, N)` tensor with the `K` axis contiguous. That is the
layout SuperGEMM takes. The unverified references predate it and read B
row-major `(K, N)`, so they are given a copy in their own layout, made once
outside the timing loop.

Times are us/call, `rel-err` is the relative Frobenius norm against the `ref` column.

In [1]:
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

import kuipy
from kuipy import unverified
from kuipy.benchmarking import bench_matrix

aten = torch.ops.aten
DEV = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Qwen2.5-0.5B-Instruct, decoding at batch 256.
HID, NH, NKV, HEAD_DIM = 896, 14, 2, 64
INTER, VOCAB, BATCH = 4864, 151936, 256
# The three attention projections are fused into one addmm with a broadcast bias.
QKV = (NH + 2 * NKV) * HEAD_DIM
SCALE = HEAD_DIM ** -0.5
ALPHA, BETA = 0.75, 1.5

_g = torch.Generator(device=DEV).manual_seed(0)

def rand(*shape, dtype=torch.bfloat16):
    return torch.randn(*shape, device=DEV, dtype=dtype, generator=_g) * 0.1

torch.cuda.get_device_name(0)

'NVIDIA RTX A6000'

In [2]:
MM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("gate_proj",   BATCH, HID,   INTER),
    ("up_proj",     BATCH, HID,   INTER),
    ("down_proj",   BATCH, INTER, HID),
    ("lm_head",     BATCH, HID,   VOCAB),
    ("square_4096", 4096,  4096,  4096),
]

# B is transposed throughout: `nn.Linear` stores its weight as (N, K) and hands
# ATen a view with K contiguous, which is the layout the SuperGEMM kernels take.
def mm_inputs(dtype):
    return lambda M, K, N: ((rand(M, K, dtype=dtype),
                             rand(N, K, dtype=dtype).t()), {})

# The unverified references take B row-major (K, N); only the Kuiper kernels
# read the transposed (N, K) weight in place. Each contender therefore gets the
# layout it is written for, and the copy is cached so it is not timed.
# Keyed on the tensor object, which keeps it alive: keying on the address would
# alias, because the caching allocator hands a freed block to the next case.
_row_major_b = {}

def rm(B):
    out = _row_major_b.get(B)
    if out is None:
        out = _row_major_b[B] = B.contiguous()
    return out

# gemm_tc is addmm-shaped; a bias-free matmul is just the absent epilogue term.
def gemm_tc_mm(A, B):
    return unverified.gemm_tc(None, A, rm(B), beta=0.0, alpha=1.0)

def gemm_tc(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_tc(C, A, rm(B), beta=beta, alpha=alpha)

def hacky_epilogue(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_hacky_epilogue(C, A, rm(B), beta=beta, alpha=alpha)

def gemm_pipe(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_pipe(C, A, rm(B), beta=beta, alpha=alpha)

def bcast_epilogue(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_bcast_bias_epilogue(C, A, rm(B), beta=beta, alpha=alpha)

def bcast_epilogue2(C, A, B, beta=1.0, alpha=1.0):
    return unverified.gemm_bcast_bias_epilogue2(C, A, rm(B), beta=beta, alpha=alpha)

MNK = lambda M, K, N: (M, K, N)
GEMM_FLOPS = lambda M, K, N: 2 * M * N * K

# A dense (M, N) epilogue term is synthetic -- the model only ever emits a
# broadcast bias -- so this is a smaller sample of the same shapes.
GEMM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("down_proj",   BATCH, INTER, HID),
    ("square_4096", 4096,  4096,  4096),
]

# What the model actually emits: the fused qkv projection, plus the two shapes
# above for comparison against the dense-C tables.
BIAS_CASES = [
    ("qkv_proj",    BATCH, HID,   QKV),
    ("o_proj",      BATCH, HID,   HID),
    ("down_proj",   BATCH, INTER, HID),
]

def addmm_inputs(dtype):
    return lambda M, K, N: ((rand(M, N, dtype=dtype), rand(M, K, dtype=dtype),
                             rand(N, K, dtype=dtype).t()),
                            {"beta": BETA, "alpha": ALPHA})

_kuiper_sdpa = kuipy.run(aten._scaled_dot_product_efficient_attention.default)

def kuiper_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    return _kuiper_sdpa(q, k, v, attn_mask, False, 0.0, is_causal, scale=scale)[0]

def cudnn_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    with sdpa_kernel(SDPBackend.CUDNN_ATTENTION):
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                              is_causal=is_causal, scale=scale,
                                              enable_gqa=True)

def attn_flops(sq, sk):
    return 4 * BATCH * NH * sq * sk * HEAD_DIM

DECODE_CASES = [(f"ctx_{c}", 1, c) for c in (128, 512, 1024, 16384)]

def decode_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"attn_mask": torch.zeros(BATCH, NH, sq, sk, device=DEV,
                                      dtype=torch.bfloat16),
             "scale": SCALE})

# Prefill: full self-attention, is_causal, no explicit mask.
PREFILL_CASES = [(f"seq_{s}", s, s) for s in (128, 512)]

def prefill_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"is_causal": True, "scale": SCALE})

## mm

`C = A @ B^T`. `gemm_tc` is addmm-shaped, so it appears here with its epilogue
term absent; the other unverified GEMMs are epilogue-only and show up in the
next section.

In [3]:
bench_matrix(MM_CASES, mm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.mm.default)),
              ("gemm_tc", gemm_tc_mm)],
             torch.mm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,gemm_tc us,ref us,kuiper rel-err,gemm_tc rel-err
0,o_proj,256,896,896,19206.123650,11275.505304,32371.612581,21.401601,36.454401,12.697600,0.000000,0.000061
1,gate_proj,256,896,4864,55587.685924,18912.272628,49367.284983,40.141439,117.985277,45.199361,0.002707,0.002707
2,up_proj,256,896,4864,52406.734056,18984.770336,50535.067610,42.577920,117.534723,44.154878,0.002708,0.002708
3,down_proj,256,4864,896,39648.326490,19192.107988,58420.160909,56.279039,116.264963,38.195200,0.002623,0.002622
4,lm_head,256,896,151936,61256.799332,62065.586817,92045.068156,1137.848282,1123.020782,757.248001,0.000000,0.000000
5,square_4096,4096,4096,4096,63726.879829,67266.939473,109224.887938,2156.687317,2043.187256,1258.311691,0.000000,0.000000


In [4]:
bench_matrix(MM_CASES, mm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.mm.default)),
              ("gemm_tc", gemm_tc_mm)],
             torch.mm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,gemm_tc us,ref us,kuiper rel-err,gemm_tc rel-err
0,o_proj,256,896,896,11702.857004,12119.806494,32794.771057,35.123200,33.914881,12.533760,0.000000,0.000029
1,gate_proj,256,896,4864,54422.374626,20776.812056,49434.482343,41.000962,107.397118,45.137920,0.000338,0.000338
2,up_proj,256,896,4864,56836.399223,20796.640269,51200.002274,39.259520,107.294722,43.581438,0.000338,0.000338
3,down_proj,256,4864,896,13042.087910,21532.332529,58326.337668,171.089916,103.628798,38.256640,0.000329,0.000329
4,lm_head,256,896,151936,51570.844330,78772.511602,92079.931874,1351.557159,884.838409,756.961288,0.000000,0.000000
5,square_4096,4096,4096,4096,71127.620318,84946.851672,108213.979667,1932.286682,1617.940521,1270.066528,0.000000,0.000000


## addmm

`D = beta*C + alpha*(A @ B)`: the full epilogue, which the bias-free `mm` path never hits.
`gemm_pipe` is fp16-only and `gemm_hacky_epilogue` bf16-only, hence the two tables.
`gemm_tc` handles both, and takes its epilogue term as an (M, N) matrix or a length-N
vector, so it is the only contender that appears in all three.

In [5]:
bench_matrix(GEMM_CASES, addmm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("hacky_epilogue", hacky_epilogue),
              ("gemm_tc", gemm_tc)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,hacky_epilogue GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,hacky_epilogue us,gemm_tc us,ref us,kuiper rel-err,hacky_epilogue rel-err,gemm_tc rel-err
0,o_proj,256,896,896,9343.761387,4941.014040,11070.461073,18687.522775,43.991041,83.189764,37.129600,21.995521,0.000003,0.000003,0.000041
1,down_proj,256,4864,896,10732.230100,5056.555236,18869.691088,51393.205671,207.912960,441.282578,118.251524,43.417602,0.002559,0.002559,0.002559
2,square_4096,4096,4096,4096,62454.153721,41440.315418,65488.695432,98647.433687,2200.637512,3316.551819,2098.666840,1393.233948,0.000000,0.000000,0.000000


In [6]:
bench_matrix(GEMM_CASES, addmm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("gemm_pipe", gemm_pipe),
              ("gemm_tc", gemm_tc)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,gemm_pipe GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,gemm_pipe us,gemm_tc us,ref us,kuiper rel-err,gemm_pipe rel-err,gemm_tc rel-err
0,o_proj,256,896,896,10913.757280,9223.529250,11911.216442,19132.889287,37.662721,44.564481,34.508801,21.483519,7.501406e-07,7.501406e-07,0.000024
1,down_proj,256,4864,896,12824.104810,11002.080471,21409.629168,51466.035518,173.998089,202.813435,104.222717,43.356161,3.186876e-04,3.186876e-04,0.000319
2,square_4096,4096,4096,4096,70628.269963,68250.002821,83031.892730,97324.108203,1945.948181,2013.757477,1655.254974,1412.177887,0.000000e+00,0.000000e+00,0.000000


### broadcast bias

`nn.Linear` emits `addmm(bias, x, W.T)` with `bias` a length-N *vector*, not an (M, N)
matrix. A tlayout is an injection, so the stride-0 row axis of a broadcast C is
inexpressible as one; C is now read as an `rotensor` over a *virtual* tensor layout,
which need not be injective, and the out-of-place epilogue reads C and writes D through
independent index functions, so dropping C's row term costs one term in the index
expression. `kuiper` is that verified path. `gemm_bcast_bias_epilogue` and
`gemm_bcast_bias_epilogue2` are the hand-edited extractions that prototyped it (the
former stages the bias slice into shared memory replicated over a fragment's rows, the
latter is the one-line lift of `TensorCore2D.To`). `kuiper+materialise` is what the
verified path had to do before: materialise the full (M, N) C and run the stock kernel.

In [7]:
def bias_inputs(dtype):
    return lambda M, K, N: ((rand(N, dtype=dtype), rand(M, K, dtype=dtype),
                             rand(N, K, dtype=dtype).t()),
                            {"beta": BETA, "alpha": ALPHA})

_kuiper_addmm = kuipy.run(aten.addmm.default)

def kuiper_dense_bias(bias, A, B, beta=1.0, alpha=1.0):
    return _kuiper_addmm(bias.expand(A.size(0), B.size(1)).contiguous(), A, B,
                         beta=beta, alpha=alpha)

bench_matrix(BIAS_CASES, bias_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", _kuiper_addmm),
              ("kuiper+materialise", kuiper_dense_bias),
              ("bcast_epilogue", bcast_epilogue),
              ("bcast_epilogue2", bcast_epilogue2),
              ("gemm_tc", gemm_tc)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,kuiper+materialise GFLOP/s,bcast_epilogue GFLOP/s,bcast_epilogue2 GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,...,kuiper+materialise us,bcast_epilogue us,bcast_epilogue2 us,gemm_tc us,ref us,kuiper rel-err,kuiper+materialise rel-err,bcast_epilogue rel-err,bcast_epilogue2 rel-err,gemm_tc rel-err
0,qkv_proj,256,896,1152,14424.903183,11613.321736,6989.382562,7349.701026,11729.454609,23739.466171,...,45.506558,75.612159,71.905279,45.056000,22.261760,0.000002,0.000002,0.000000,0.000002,0.000023
1,o_proj,256,896,896,11449.173105,8968.006971,5428.834222,5507.793572,11897.095154,18601.465498,...,45.834241,75.714560,74.629121,34.549761,22.097280,0.000003,0.000003,0.000000,0.000003,0.000025
2,down_proj,256,4864,896,13017.156608,12487.519177,5306.268169,5436.823102,21276.103346,50794.218287,...,178.687992,420.515823,410.417938,104.876804,43.929601,0.000319,0.000319,0.000319,0.000319,0.000319


## sdpa

The decode mask is a dense `(B, Hq, Sq, Sk)` tensor: the Kuiper mask is read through a
`rotensor`, whose layout need not be an injection, but the instantiation template only
emits the dense row-major layout, so the mask is materialised and handed to every
contender for fairness. Prefill passes no mask at all, which the kernel selects with a
broadcast layout plus `has_mask = false`.

In [8]:
bench_matrix(DECODE_CASES, decode_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("kuiper", kuiper_sdpa),
              ("manual_extract", unverified.flash_attn_manual_extract),
              ("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)

,case,Sq,Sk,kuiper GFLOP/s,manual_extract GFLOP/s,fa1 GFLOP/s,fa2 GFLOP/s,ref GFLOP/s,kuiper us,manual_extract us,fa1 us,fa2 us,ref us,kuiper rel-err,manual_extract rel-err,fa1 rel-err,fa2 rel-err
0,ctx_128,1,128,744.727277,756.317595,824.144882,2230.416164,2445.373209,157.695999,155.279360,142.499838,52.654080,48.025599,0.002327,0.002327,0.002327,0.001661
1,ctx_512,1,512,1037.668617,1048.240589,1030.116281,2658.507317,3788.833830,452.709122,448.143349,456.028175,176.701431,123.985920,0.002350,0.002350,0.002350,0.001620
2,ctx_1024,1,1024,1191.316037,1186.724316,1116.049185,2765.232201,3906.599751,788.643875,791.695328,841.830368,339.763184,240.496635,0.002360,0.002360,0.002360,0.001465
3,ctx_16384,1,16384,1531.496077,1534.040713,1389.260357,3958.961932,4034.869030,9815.490723,9799.208984,10820.423584,3797.052307,3725.619202,0.002323,0.002323,0.002323,0.000878


In [9]:
bench_matrix(PREFILL_CASES, prefill_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("kuiper", kuiper_sdpa),
              ("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)

,case,Sq,Sk,kuiper GFLOP/s,fa1 GFLOP/s,fa2 GFLOP/s,ref GFLOP/s,kuiper us,fa1 us,fa2 us,ref us,kuiper rel-err,fa1 rel-err,fa2 rel-err
0,seq_128,128,128,3484.915311,5226.565390,13977.019871,64869.923483,4313.558350,2876.149902,1075.507202,231.731205,0.001299,0.000927,0.000927
1,seq_512,512,512,7364.925563,9532.609682,35357.567344,112435.985509,32657.243652,25231.093750,6802.452393,2139.156494,0.001560,0.001074,0.001074
